# Barcode Detection with pyzbar

This notebook tests `pyzbar` as an atomic decoder against the 300 DPI rendered page cache. `pyzbar` wraps the native `zbar` library, so the decode call runs in a subprocess: if the local `zbar`/Python combo crashes, the notebook records that instead of killing the kernel.


In [ ]:
from pathlib import Path
import json
import shutil
import math
import os
import subprocess
import sys

from PIL import Image, ImageDraw

try:
    from IPython.display import display
except ImportError:
    def display(image):
        return image


def find_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / 'data' / 'pdfs').exists():
            return path
    raise RuntimeError('Could not find repo root containing data/pdfs')


ROOT = find_root()
PDF_PATH = ROOT / 'data' / 'pdfs' / 'private-evaluation-document.pdf'
PAGES = list(range(1, 22))
OUT_DIR = ROOT / 'output' / 'pyzbar-atom'
PAGE_DIR = OUT_DIR / 'rendered-pages'
DPI = 300
OUT_DIR.mkdir(parents=True, exist_ok=True)

ZBAR_LIB = next((p for p in [
    Path('/opt/homebrew/lib/libzbar.dylib'),
    Path('/usr/local/lib/libzbar.dylib'),
] if p.exists()), None)

RUNNER = r'''
import json
import sys
from PIL import Image
from pyzbar.pyzbar import decode

items = []
for result in decode(Image.open(sys.argv[1])):
    rect = result.rect
    polygon = [{'x': p.x, 'y': p.y} for p in result.polygon]
    items.append({
        'type': result.type,
        'raw_value': result.data.decode('utf-8', errors='replace'),
        'rect': {'x': rect.left, 'y': rect.top, 'w': rect.width, 'h': rect.height},
        'polygon': polygon,
    })
print(json.dumps({'ok': True, 'detections': items}))
'''


def pyzbar_env():
    env = os.environ.copy()
    if ZBAR_LIB:
        lib_dir = str(ZBAR_LIB.parent)
        env['DYLD_LIBRARY_PATH'] = lib_dir + os.pathsep + env.get('DYLD_LIBRARY_PATH', '')
        env['DYLD_FALLBACK_LIBRARY_PATH'] = lib_dir + os.pathsep + env.get('DYLD_FALLBACK_LIBRARY_PATH', '')
    return env


def decode_page(page_path):
    proc = subprocess.run(
        [sys.executable, '-c', RUNNER, str(page_path)],
        capture_output=True,
        text=True,
        env=pyzbar_env(),
    )
    if proc.returncode != 0:
        return {
            'ok': False,
            'returncode': proc.returncode,
            'stderr': proc.stderr.strip(),
        }
    return json.loads(proc.stdout)


def draw_page(page_path, detections):
    image = Image.open(page_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    for det in detections:
        pts = [(p['x'], p['y']) for p in det.get('polygon', [])]
        if pts:
            draw.line(pts + [pts[0]], fill='red', width=8)
        else:
            r = det['rect']
            draw.rectangle([r['x'], r['y'], r['x'] + r['w'], r['y'] + r['h']], outline='red', width=8)
    return image


print(f'Python: {sys.executable}')
print(f'zbar: {ZBAR_LIB or "not found"}')

def find_pdftoppm():
    found = shutil.which('pdftoppm')
    if found:
        return found
    for candidate in [Path('/opt/homebrew/bin/pdftoppm'), Path('/usr/local/bin/pdftoppm')]:
        if candidate.exists():
            return str(candidate)
    cache_root = Path.home() / '.cache' / 'codex-runtimes'
    if cache_root.exists():
        for candidate in cache_root.glob('**/pdftoppm'):
            if candidate.is_file():
                return str(candidate)
    raise RuntimeError('pdftoppm not found. Install Poppler or add pdftoppm to PATH.')


def render_pdf_if_needed():
    PAGE_DIR.mkdir(parents=True, exist_ok=True)
    marker = PAGE_DIR / '.dpi'
    pages_ready = all((PAGE_DIR / f'page-{page:02d}.png').exists() for page in PAGES)
    dpi_ready = marker.exists() and marker.read_text().strip() == str(DPI)
    if pages_ready and dpi_ready:
        return
    for old_page in PAGE_DIR.glob('page-*.png'):
        old_page.unlink()
    subprocess.run(
        [find_pdftoppm(), '-r', str(DPI), '-png', str(PDF_PATH), str(PAGE_DIR / 'page')],
        check=True,
    )
    marker.write_text(str(DPI), encoding='utf-8')



Run `pyzbar` across the 300 DPI rendered pages and save the raw result. On this machine, the current Python 3.13 + Homebrew zbar combination returns `-11` for each subprocess, which means a native segfault inside `pyzbar.decode()`.


In [ ]:
render_pdf_if_needed()
results = []
for page in PAGES:
    page_path = PAGE_DIR / f'page-{page:02d}.png'
    result = decode_page(page_path)
    detections = result.get('detections', [])
    results.append({
        'page': page,
        'image': str(page_path),
        'ok': result.get('ok', False),
        'decoded_count': len(detections),
        'detections': detections,
        'error': None if result.get('ok') else result,
    })
    status = f'decoded={len(detections)}' if result.get('ok') else f'failed rc={result.get("returncode")}'
    print(f'page {page:02d}: {status}')

json_path = OUT_DIR / 'detections.json'
json_path.write_text(json.dumps(results, indent=2), encoding='utf-8')
print(f'Saved JSON: {json_path}')


Draw any successful detections. If the grid is empty, `pyzbar` did not reach a clean decode on this runtime.


In [ ]:
thumbs = []
for item in results:
    if item['detections']:
        page_path = Path(item['image'])
        preview = draw_page(page_path, item['detections'])
        preview.thumbnail((600, 800))
        thumbs.append((item['page'], preview.copy()))

if thumbs:
    cols = 2
    w = max(img.width for _, img in thumbs)
    h = max(img.height for _, img in thumbs)
    grid = Image.new('RGB', (cols * w, math.ceil(len(thumbs) / cols) * h), 'white')
    for idx, (_, img) in enumerate(thumbs):
        grid.paste(img, ((idx % cols) * w, (idx // cols) * h))
    grid_path = OUT_DIR / 'grid.png'
    grid.save(grid_path)
    print(f'Saved grid: {grid_path}')
    display(grid)
else:
    print('No successful pyzbar detections to draw.')